# Tópico modelo — exemplo executável

**Guia Aberto de Arquitetura de Software** · [Página do tópico](https://ORG.github.io/REPO/content/00-modelo/)

Autor: Nome Completo do Autor · Licença: CC BY-NC 4.0

---

Este notebook é o exemplo executável obrigatório do tópico. Ele precisa:

1. **Rodar de ponta a ponta em ambiente limpo** — teste no Colab antes de abrir o PR
2. **Demonstrar o conceito**, não apenas ilustrá-lo
3. **Mostrar a violação** da restrição arquitetural e sua consequência
4. **Explicar em células de texto**, não só em comentários de código

> Substitua todo o conteúdo abaixo. A estrutura das seções é o que deve ser mantido.

## 0. Dependências

Declare tudo o que o notebook precisa na primeira célula de código. No Colab,
prefira `pip install -q`.

In [ ]:
# Sem dependências externas neste modelo.
# Se o seu exemplo precisar de bibliotecas, instale aqui:
# !pip install -q pandas matplotlib

import sys
from abc import ABC, abstractmethod
from dataclasses import dataclass, field

print(f"Python {sys.version.split()[0]}")

## 1. O problema

Descreva em texto qual dificuldade concreta o código abaixo vai demonstrar.
Conecte ao mesmo exemplo usado no `index.qmd` — se o texto discute um sistema de
pedidos, o notebook implementa um sistema de pedidos.

## 2. Implementação seguindo o estilo

Aqui o conceito aplicado corretamente. Comente as **decisões**, não a sintaxe.

In [ ]:
# Camada de domínio: não conhece nada acima dela.
@dataclass
class Pedido:
    id: int
    itens: list = field(default_factory=list)

    def total(self) -> float:
        return sum(preco for _, preco in self.itens)


# Porta (interface) declarada pelo domínio, implementada pela infraestrutura.
# Essa inversão é o que impede a dependência de apontar para fora do domínio.
class RepositorioPedidos(ABC):
    @abstractmethod
    def salvar(self, pedido: Pedido) -> None: ...

    @abstractmethod
    def buscar(self, pedido_id: int) -> Pedido | None: ...


# Camada de infraestrutura: implementa a porta, e o domínio não a conhece.
class RepositorioMemoria(RepositorioPedidos):
    def __init__(self):
        self._dados: dict[int, Pedido] = {}

    def salvar(self, pedido: Pedido) -> None:
        self._dados[pedido.id] = pedido

    def buscar(self, pedido_id: int) -> Pedido | None:
        return self._dados.get(pedido_id)


# Camada de aplicação: orquestra, sem regra de negócio própria.
class ServicoPedidos:
    def __init__(self, repositorio: RepositorioPedidos):
        self.repositorio = repositorio

    def criar(self, pedido_id: int, itens: list) -> Pedido:
        pedido = Pedido(id=pedido_id, itens=itens)
        self.repositorio.salvar(pedido)
        return pedido


servico = ServicoPedidos(RepositorioMemoria())
pedido = servico.criar(1, [("livro", 89.90), ("caneta", 12.50)])
print(f"Pedido {pedido.id} — total: R$ {pedido.total():.2f}")

### Por que isso é testável

A consequência prática de respeitar a restrição: o domínio pode ser exercitado sem
infraestrutura real.

In [ ]:
class RepositorioFalso(RepositorioPedidos):
    """Dublê de teste — possível justamente porque o domínio depende da porta,
    não de uma implementação concreta."""

    def __init__(self):
        self.salvos = []

    def salvar(self, pedido: Pedido) -> None:
        self.salvos.append(pedido)

    def buscar(self, pedido_id: int) -> Pedido | None:
        return next((p for p in self.salvos if p.id == pedido_id), None)


falso = RepositorioFalso()
ServicoPedidos(falso).criar(2, [("teclado", 250.00)])

assert len(falso.salvos) == 1
assert falso.salvos[0].total() == 250.00
print("Teste passou — domínio exercitado sem banco de dados.")

## 3. A violação

Agora mostre o que acontece quando a restrição do estilo é quebrada. Esta seção é
obrigatória: é ela que transforma o notebook em demonstração, e não em ilustração.

In [ ]:
# ANTIPADRÃO: o domínio passa a conhecer a infraestrutura diretamente.
# A dependência agora aponta para fora, e a restrição do estilo foi violada.

class PedidoAcoplado:
    def __init__(self, id: int, itens: list):
        self.id = id
        self.itens = itens

    def salvar(self):
        # O domínio instancia infraestrutura concreta.
        # Não há como testar esta classe sem carregar o repositório real.
        repo = RepositorioMemoria()
        repo.salvar(self)  # type: ignore[arg-type]
        return repo


print("Consequência: PedidoAcoplado não pode ser testado em isolamento,")
print("e trocar o mecanismo de persistência exige editar a regra de negócio.")

## 4. Consequência mensurável

Quantifique o custo da violação de alguma forma verificável: contagem de
dependências, tempo de execução, número de arquivos que precisam mudar. Uma
medida simples e honesta vale mais que uma afirmação genérica.

In [ ]:
import dis

INFRA = {"RepositorioMemoria", "RepositorioFalso"}

def dependencias_concretas(cls) -> list[str]:
    """Procura, no bytecode dos métodos, referências a classes concretas de
    infraestrutura. Funciona em qualquer ambiente, sem depender do arquivo-fonte."""
    encontradas = set()
    for nome, membro in vars(cls).items():
        codigo = getattr(membro, "__code__", None)
        if codigo is None:
            continue
        for instr in dis.get_instructions(codigo):
            if instr.opname in ("LOAD_GLOBAL", "LOAD_NAME") and instr.argval in INFRA:
                encontradas.add(instr.argval)
    return sorted(encontradas)

for cls in (Pedido, ServicoPedidos, PedidoAcoplado):
    deps = dependencias_concretas(cls)
    situacao = "viola o estilo" if deps else "respeita o estilo"
    rotulo = ", ".join(deps) if deps else "nenhuma"
    print(f"{cls.__name__:18} dependencias concretas: {rotulo:22} -> {situacao}")

## 5. Conclusão

Amarre o resultado ao trade-off discutido no texto. Uma ou duas frases.

---

### Referências

Use a mesma numeração da página do tópico, para que `[1]` signifique a mesma coisa
nos dois lugares.

[1] Bass, L., Clements, P., Kazman, R. *Software Architecture in Practice*, 4ª ed. Addison-Wesley, 2021.

[2] Parnas, D. L. "On the Criteria To Be Used in Decomposing Systems into Modules". *CACM* 15(12), 1972. DOI 10.1145/361598.361623.

---

Conteúdo sob CC BY-NC 4.0 — uso livre com crédito. Código sob MIT.